<a href="https://colab.research.google.com/github/Najma-1437/Knowledge-Engineering/blob/main/Autonomous_Trailer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Practical Exercise 4: Developing a Knowledge-Based Agent

### Knowledge-Based Autonomous Trailer

## Identify the Task

The agent should use knowledge about the road network to choose a route between Nairobi and Kisumu while respecting road availability.

- The agent's environment: a simplified directed Kenyan highway network.
- The goal: identify a minimum-distance permitted route.
- The inputs: current location, destination, road connections and road conditions.
- The outputs: selected route, total distance and an explanation of the decision.

## Assemble Relevant Knowledge


In [1]:
#Highway Network

highway_graph = {
    "Nairobi": {
        "Naivasha": 90,
        "Narok": 142,
        "Thika": 40
    },
    "Naivasha": {
        "Nakuru": 71
    },
    "Narok": {
        "Bomet": 81
    },
    "Thika": {
        "Nyeri": 110
    },
    "Nyeri": {
        "Nyahururu": 103
    },
    "Nyahururu": {
        "Nakuru": 75
    },
    "Nakuru": {
        "Kericho": 108
    },
    "Bomet": {
        "Kericho": 79
    },
    "Kericho": {
        "Kisumu": 84
    },
    "Kisumu": {}
}

# Display the highway network
print("=== HIGHWAY NETWORK KNOWLEDGE BASE ===")

for town, connections in highway_graph.items():
    if connections:
        for destination, distance in connections.items():
            print(f"{town} -> {destination}: {distance} km")
    else:
        print(f"{town}: No outgoing connections")

# Verify the graph
number_of_towns = len(highway_graph)

number_of_roads = sum(
    len(connections)
    for connections in highway_graph.values()
)

print("\n=== GRAPH SUMMARY ===")
print("Number of towns:", number_of_towns)
print("Number of directed roads:", number_of_roads)


=== HIGHWAY NETWORK KNOWLEDGE BASE ===
Nairobi -> Naivasha: 90 km
Nairobi -> Narok: 142 km
Nairobi -> Thika: 40 km
Naivasha -> Nakuru: 71 km
Narok -> Bomet: 81 km
Thika -> Nyeri: 110 km
Nyeri -> Nyahururu: 103 km
Nyahururu -> Nakuru: 75 km
Nakuru -> Kericho: 108 km
Bomet -> Kericho: 79 km
Kericho -> Kisumu: 84 km
Kisumu: No outgoing connections

=== GRAPH SUMMARY ===
Number of towns: 10
Number of directed roads: 11


### Sources of Knowledge
The network consists of 10 towns and 11 directed road connections, with distances measured in kilometres. These distances serve as the edge costs used during route planning.

The graph provides the agent with information about:

- Towns
Road connections
- Distances
- Initial and goal locations

### Assumptions

1. Directed road network: a simplification of the highway network and does not necessarily indicate that the actual roads are one-way.

2. Fixed distances: The distance between two connected towns is assumed to remain constant during route planning.

3. Road availability: All represented roads are initially assumed to be open. Road closures will be introduced as simulated facts to demonstrate knowledge updates and reasoning.

4. Non-negative costs: Every road has a non-negative distance, allowing Uniform Cost Search to determine a minimum-cost route.

5. Simplified environment: The initial knowledge base does not account for live traffic congestion, weather conditions, road accidents or vehicle-specific restrictions.

6. Destination: The autonomous trailer's initial task is to navigate from Nairobi to Kisumu using the available highway network.

The initial implementation uses the highway graph and distance measurements from Practical Exercise 3, Part 1.

No external APIs or web scraping tools are used to construct the initial knowledge base.

In [2]:
print("Number of towns:", len(highway_graph))
print(
    "Number of directed roads:",
    sum(len(neighbors) for neighbors in highway_graph.values())
)

Number of towns: 10
Number of directed roads: 11


##Define the Vocabulary

### Consants
They identify specific objects in the domain. For example, Nairobi, Naivasha, Kericho

### Predicates

| Predicate | Interpretation |
| :--- | :--- |
| `Connected(u, v)` | A directed road connects town *u* to town *v* |
|`Distance(u, v, d)` | The road from *u* to *v* has distance *d* |
| `RoadOpen(u, v)` | The connection from *u* to *v* is open |
|`Usable(u, v)` | The trailer is permitted to travel from *u* to *v* |
| `At(trailer, u)` | The trailer is currently at town *u* |
|`Goal(u)` | Town *u* is the intended destination |


In [3]:
# Initial and goal states
initial_state = "Nairobi"
goal_state = "Kisumu"

print("Initial state:", initial_state)
print("Goal state:", goal_state)

Initial state: Nairobi
Goal state: Kisumu


## Encode General Knowledge About the Domain


### General Rules

| Rule | Meaning |
| :--- | :--- |
| `R1` | A road can be considered only if it exists in the directed graph |
| `R2` | A road can be used only if its status is open |
| `R3` | Every road's traversal cost is its stored distance |
| `R4` | A route's cost is the sum of its road distances |
| `R5` | The agent should select a minimum-cost route among permitted routes |

The central logical rule is:

$$
\text{Connected}(u,v) \land \text{RoadOpen}(u,v) \Rightarrow \text{Usable}(u,v)
$$


in other words, If a directed road connects two towns and that road is open, the trailer may use that connection.


In [6]:
#Road Availability Facts
# All roads are assumed open for the baseline simulation

road_status = {
    (source, destination): "open"
    for source, neighbors in highway_graph.items()
    for destination in neighbors
}
print(road_status[("Nairobi", "Naivasha")])
print("Road status entries:", len(road_status))


open
Road status entries: 11


In [7]:
#Knowledge Base Validation
# Check that the initial and goal towns exist
assert initial_state in highway_graph
assert goal_state in highway_graph

# Check road facts
for source, neighbors in highway_graph.items():
    for destination, distance in neighbors.items():
        assert destination in highway_graph
        assert distance >= 0

# Check that every road has a status
for source, neighbors in highway_graph.items():
    for destination in neighbors:
        assert (source, destination) in road_status

# Check valid road statuses
for status in road_status.values():
    assert status in {"open", "closed"}

print("Knowledge base validation successful.")

Knowledge base validation successful.
